In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import sys; sys.path.append("..")
from src.data.harmonise import prepare_fitzpatrick
from pathlib import Path

In [ ]:
RAW = Path("..") / "data" / "raw"
for folder in sorted(RAW.iterdir()):
    print(folder.name, "→", len(list(folder.rglob("*"))), "files")

In [ ]:
df = prepare_fitzpatrick(RAW / "fitzpatrick17k" / "fitzpatrick17ku.csv",
                                 RAW / "fitzpatrick17k")
df.shape

In [ ]:
df.head()

In [ ]:
strat_key = df["skin_group"] + "_" + df["target"].astype(str)
print(strat_key)

In [ ]:
temp_df, test_df = train_test_split(df, test_size = 0.15, random_state  = 42, stratify = strat_key)
print(temp_df.shape)
print(test_df.shape)

In [ ]:
strat_key_temp = temp_df["skin_group"] + "_" + temp_df["target"].astype(str)
print(strat_key_temp)

In [ ]:
train_df, val_df = train_test_split(temp_df, test_size = 0.15 / (1 - 0.15), random_state  = 42, stratify = strat_key_temp)
print(train_df.shape)
print(val_df.shape)

In [ ]:
train_df = train_df.assign(split = "train")
val_df   = val_df.assign(split = "val")
test_df  = test_df.assign(split = "test")

print(train_df.shape)

In [ ]:
new_df = (
    pd.concat([train_df, val_df, test_df])
      .sort_index()          # back to original row order
      .reset_index(drop=True)
)

In [ ]:
new_df["split"].value_counts()

In [ ]:
pd.crosstab([new_df["skin_group"], new_df["target"]], new_df["split"])

In [ ]:
train_ids, val_ids, test_ids = (set(df["group_id"]) for df in (train_df, val_df, test_df))

overlaps = {
    "train∩val":  train_ids & val_ids,
    "train∩test": train_ids & test_ids,
    "val∩test":   val_ids & test_ids,
}

assert all(len(v) == 0 for v in overlaps.values()), \
    "Leakage detected: " + ", ".join(f"{k}={sorted(v)[:10]}" for k, v in overlaps.items() if v)

print("✓ No group_id leakage across splits")

In [ ]:
new_df[["group_id", "split"]].sort_values(["split", "group_id"]).to_csv(
    "../splits/fitzpatrick17k_splits.csv", index = False
)

In [ ]:
from src.data.harmonise import prepare_fitzpatrick
from src.data.splits import make_splits, save_splits
from src.config import RAW, RANDOM_SEED, VAL_FRAC, TEST_FRAC, SPLITS_DIR

In [ ]:
csv_path = RAW / "fitzpatrick17k" / "fitzpatrick17ku.csv"
img_dir = RAW / "fitzpatrick17k"
fitz_clean = prepare_fitzpatrick(csv_path, img_dir)
print(fitz_clean.shape)

In [ ]:
fitz_split = make_splits(fitz_clean, RANDOM_SEED, VAL_FRAC, TEST_FRAC)

In [ ]:
print(fitz_split["split"].value_counts())

In [ ]:
save_splits(fitz_split, SPLITS_DIR / "fitzpatrick17k_splits.csv")

In [ ]:
ct = pd.crosstab([fitz_split["skin_group"], fitz_split["target"]], fitz_split["split"])
ct.reindex(["light", "medium", "dark"], level=0)[["train", "val", "test"]]

In [ ]:
from src.data.transforms import get_transforms
from src.data.dataset import SkinLesionDataset
from torch.utils.data import DataLoader
import torch

In [ ]:
train_ds = SkinLesionDataset(train_df, get_transforms(train=True))
img, target, group = train_ds[0]
print(len(train_ds), img.shape, img.dtype, target, group)

In [ ]:
loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
imgs, targets, groups = next(iter(loader))
print(imgs.shape, targets.shape, len(groups))

In [ ]:
eval_ds = SkinLesionDataset(train_df, get_transforms(train=False))
c, *_ = eval_ds[0]
d, *_ = eval_ds[0]
print("eval stable:", torch.equal(c, d))   # expect True

a, *_ = train_ds[0]
b, *_ = train_ds[0]
print("train varies:", not torch.equal(a, b))  # expect True

In [ ]:
splits = pd.read_csv(SPLITS_DIR / "fitzpatrick17k_splits.csv")
fitz = fitz_clean.merge(splits, on="group_id", validate="one_to_one")
assert len(fitz) == 4320
print(fitz["split"].value_counts())

In [ ]:
print(len(train_df), len(val_df), len(test_df), len(train_df)+len(val_df)+len(test_df))
print(fitz["split"].value_counts(normalize=True))

In [ ]:
from src.config import MODEL_NAME
from src.models.model import build_model, get_device

model = build_model(MODEL_NAME)
print(sum(p.numel() for p in model.parameters()))

In [ ]:
device = get_device()
model = model.to(device).eval()
with torch.no_grad():
    out = model(imgs.to(device))
print(device, out.shape, torch.sigmoid(out).squeeze()[:5])

In [ ]:
print(model.pretrained_cfg["mean"], model.pretrained_cfg["std"])

In [ ]:
from src.config import CHECKPOINT_DIR, RESULTS_DIR

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from torch.utils.data import DataLoader, Subset
from src.models.train import train_one_epoch
from src.models.model import build_model
from src.utils import set_seed

set_seed(42)
device = get_device()
tiny_loader = DataLoader(Subset(train_ds, range(64)), batch_size=32, shuffle=True)

m = build_model(MODEL_NAME).to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-2)
loss_fn = torch.nn.BCEWithLogitsLoss()

for i in range(20):
    loss = train_one_epoch(m, tiny_loader, loss_fn, opt, device)
    if i % 4 == 0 or i == 19:
        print(i, f"{loss:.4f}")

In [ ]:
from src.config import RAW, SPLITS_DIR
from src.data.harmonise import prepare_fitzpatrick
import pandas as pd

fitz_clean = prepare_fitzpatrick(
    csv_path=RAW / "fitzpatrick17k" / "fitzpatrick17ku.csv",
    img_dir=RAW / "fitzpatrick17k",
)

splits = pd.read_csv(SPLITS_DIR / "fitzpatrick17k_splits.csv")
fitz = fitz_clean.merge(splits, on="group_id", validate="one_to_one")
train_df = fitz[fitz["split"] == "train"].reset_index(drop=True)

print(fitz.columns.tolist())
print(train_df["image_path"].iloc[0])
print(train_df["target"].value_counts())
print(train_df["skin_group"].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.config import RESULTS_DIR, MODEL_NAME, RANDOM_SEED

# --- load history ------------------------------------------------------------
hist = pd.read_csv(RESULTS_DIR / f"{MODEL_NAME}_seed{RANDOM_SEED}_history.csv")

best_epoch = int(hist.loc[hist["val_auc"].idxmax(), "epoch"])
best_auc   = hist["val_auc"].max()
print(f"best epoch: {best_epoch} | best val AUC: {best_auc:.4f}")

# --- figure ------------------------------------------------------------------
fig, (ax_loss, ax_auc) = plt.subplots(1, 2, figsize=(10, 3.5))

# Left: losses
ax_loss.plot(hist["epoch"], hist["train_loss"], marker="o", ms=3,
             label="train loss", color="C0")
ax_loss.plot(hist["epoch"], hist["val_loss"], marker="s", ms=3,
             label="val loss", color="C1")
ax_loss.axvline(best_epoch, color="grey", ls="--", lw=1)
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("BCE loss")
ax_loss.set_title("Loss")
ax_loss.legend(frameon=False)
ax_loss.grid(alpha=0.3)

# Right: AUC
ax_auc.plot(hist["epoch"], hist["val_auc"], marker="o", ms=3,
            label="val AUC", color="C2")
ax_auc.axvline(best_epoch, color="grey", ls="--", lw=1,
               label=f"best (epoch {best_epoch})")
ax_auc.set_xlabel("epoch")
ax_auc.set_ylabel("AUC")
ax_auc.set_title("Validation AUC")
ax_auc.set_ylim(0.5, 1.0)
ax_auc.legend(frameon=False, loc="lower right")
ax_auc.grid(alpha=0.3)

fig.suptitle(f"{MODEL_NAME} — seed {RANDOM_SEED}", y=1.02, fontsize=10)
fig.tight_layout()

out = RESULTS_DIR / f"{MODEL_NAME}_seed{RANDOM_SEED}_learning_curve.pdf"
fig.savefig(out, bbox_inches="tight", dpi=300)
fig.savefig(out.with_suffix(".png"), bbox_inches="tight", dpi=300)
print(f"saved -> {out}")
plt.show()